# Week 5 — Improve the first models

In [18]:
import zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.calibration import CalibrationDisplay
import joblib

zip_path = "csv.zip"
random_state = 0
max_rows_per_year = None
cpu_threads = 8
thresholds = 0.2

with zipfile.ZipFile(zip_path) as z:
    files = {
        Path(name).name: name
        for name in z.namelist()
        if name.lower().endswith(".csv")
    }

numeric_features = [
    "bmi_00",
    "mental_health_days_00",
    "physical_health_days_00"
]

categorical_features = [
    "age_00", "sex_00", "race_00", "marital_status_00", "education_00",
    "income", "gen_health_00", "exercise_00", "smoked_100_cigarettes_00",
    "drinks_alcohol_00", "had_stroke_00", "had_heart_attack_00",
    "had_coronary_heart_disease_00", "cost_barrier_00",
    "l_checkup_00", "has_personal_doctor_00"
]

predictor_columns = numeric_features + categorical_features

target_map = {
    "Diabetic": 1,
    "Non-diabetic": 0,
    "Pre-diabetic": 0
}

In [2]:
annual_frames = []

with zipfile.ZipFile(zip_path) as z:
    for year in range(2005, 2021):
        annual_name = "DATASET_" + str(year) + ".csv"
        with z.open(files[annual_name]) as f:
            raw = pd.read_csv(f, low_memory=False)

        # Records need a known target for supervised learning
        known_target = raw["diabetes"].notna()
        clean = raw.loc[known_target].copy()
        clean["target"] = clean["diabetes"].map(target_map)
        clean["target"] = clean["target"].astype("int8")

        # This loop uses only 2005-2020, which has the older income column
        income_source = "income_00"
        maximum_income_code = 8

        original_income = clean[income_source]
        income = pd.to_numeric(original_income, errors="coerce")
        valid_income = income.isin(range(1, maximum_income_code + 1))
        income = income.where(valid_income)
        clean["income"] = income

        # Keep the record, but replace BMI outside the study range with missing
        bmi_raw = clean["bmi_00"]
        bmi = pd.to_numeric(bmi_raw, errors="coerce")
        valid_bmi = bmi.between(12.00, 99.99)
        clean["bmi_00"] = bmi.where(valid_bmi)

        # Health-day counts must stay between 0 and 30
        for feature in ["mental_health_days_00", "physical_health_days_00"]:
            original_values = clean[feature]
            values = pd.to_numeric(original_values, errors="coerce")
            clean[feature] = values.where(values.between(0, 30))

        # Keep only the selected predictors, target, and split year
        clean["year"] = year
        selected_columns = predictor_columns + ["target", "year"]
        clean = clean[selected_columns].copy()

        # Use matching text labels across years; keep missing as its own label
        for feature in categorical_features:
            labels = clean[feature].astype("string")
            labels = labels.str.replace(r"\.0$", "", regex=True)
            labels = labels.fillna("<missing>")
            clean[feature] = labels.astype(str)

        # Smaller numeric types reduce memory use
        for feature in numeric_features:
            clean[feature] = clean[feature].astype("float32")

        # Select a reproducible annual sample without balancing the two classes
        if max_rows_per_year is not None:
            if len(clean) > max_rows_per_year:
                clean, unused_rows = train_test_split(
                    clean,
                    train_size=max_rows_per_year,
                    stratify=clean["target"],
                    random_state=random_state
                )
                del unused_rows

        annual_frames.append(clean)
        print("Prepared", year, "— selected rows:", len(clean))
        del raw, clean

model_data = pd.concat(annual_frames, ignore_index=True)
del annual_frames

Prepared 2005 — selected rows: 355446
Prepared 2006 — selected rows: 354497
Prepared 2007 — selected rows: 429942
Prepared 2008 — selected rows: 412898
Prepared 2009 — selected rows: 431636
Prepared 2010 — selected rows: 449156
Prepared 2011 — selected rows: 503846
Prepared 2012 — selected rows: 473243
Prepared 2013 — selected rows: 490368
Prepared 2014 — selected rows: 461879
Prepared 2015 — selected rows: 439881
Prepared 2016 — selected rows: 483230
Prepared 2017 — selected rows: 447952
Prepared 2018 — selected rows: 434232
Prepared 2019 — selected rows: 416661
Prepared 2020 — selected rows: 399050


In [15]:
# Use all 2005-2020 records for final training
X_train = model_data[predictor_columns].copy()
y_train = model_data["target"].copy()

print("Training rows:", len(X_train))
print("Diabetic percentage:", round(y_train.mean() * 100, 3))

Training rows: 6983917
Diabetic percentage: 12.514


In [16]:
# Learn the numeric preprocessing from the full 2005-2020 training data
numeric_imputer = SimpleImputer(strategy="median")
numeric_scaler = StandardScaler()

numeric_steps = [
    ("imputer", numeric_imputer),
    ("scaler", numeric_scaler)
]

numeric_pipeline = Pipeline(steps=numeric_steps)

# Convert categorical features to one-hot columns
# Unknown categories in future years will be ignored
category_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True,
    dtype=np.float32
)

# Apply numeric preprocessing and categorical encoding
transformer_steps = [
    ("numeric", numeric_pipeline, numeric_features),
    ("category", category_encoder, categorical_features)
]

preprocessor = ColumnTransformer(
    transformers=transformer_steps,
    sparse_threshold=1.0
)

# Fit preprocessing on 2005-2020 only
X_train_ready = preprocessor.fit_transform(X_train)
X_train_ready = X_train_ready.astype("float32")

print("Training matrix shape:", X_train_ready.shape)
print("Encoded columns:", len(preprocessor.get_feature_names_out()))

Training matrix shape: (6983917, 89)
Encoded columns: 89


## Models


In [19]:
lr_final = LogisticRegression(
    max_iter=1000,
    solver="lbfgs",
    random_state=random_state
)

lr_final.fit(X_train_ready, y_train)

print("Logistic Regression finished.")
print("Iterations used:", lr_final.n_iter_)

Logistic Regression finished.
Iterations used: [44]


In [20]:
rf_final = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    min_samples_leaf=20,
    n_jobs=cpu_threads,
    random_state=random_state
)

rf_final.fit(X_train_ready, y_train)

print("Random Forest finished.")

Random Forest finished.


In [21]:
xgb_final = XGBClassifier(
    n_estimators=150,
    max_depth=4,
    learning_rate=0.05,
    subsample=1.0,
    colsample_bytree=1.0,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    n_jobs=cpu_threads,
    random_state=random_state
)

xgb_final.fit(X_train_ready, y_train)

print("XGBoost finished.")

XGBoost finished.


In [23]:
joblib.dump(thresholds, "final_threshold.joblib")
joblib.dump(preprocessor, "final_preprocessor.joblib")
joblib.dump(lr_final, "final_logistic_regression.joblib")
joblib.dump(rf_final, "final_random_forest.joblib")
joblib.dump(xgb_final, "final_xgboost.joblib")

print("Final models and preprocessor saved.")

Final models and preprocessor saved.
